
# gffcompare details

This notebook loads per-direction class_code counts from `qc_metrics/<assembly_accession>/gffcompare/class_counts_<direction>.tsv` and produces summary plots.

- Directions are treated independently: `Ensembl_to_CAT` and `CAT_to_Ensembl` have distinct denominators (query transcripts).
- Figures are saved under `results/figures/`.


In [ ]:

import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

QC_DIR = Path('results/qc_metrics')  # adjust if needed
FIG_DIR = Path('results/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Discover class_counts files
files = list(QC_DIR.glob('*/gffcompare/class_counts_*.tsv'))
print(f"Found {len(files)} class_count files")

cols = [
    'assembly_accession','sample_name','direction','class_code','n_transcripts','denominator','pct'
]

dfs = []
for f in files:
    try:
        df = pd.read_csv(f, sep='	')
        # Validate columns and coerce types
        missing = [c for c in cols if c not in df.columns]
        if missing:
            print(f"Skipping {f}: missing columns {missing}")
            continue
        df = df[cols].copy()
        df['n_transcripts'] = pd.to_numeric(df['n_transcripts'], errors='coerce').fillna(0).astype(int)
        df['denominator'] = pd.to_numeric(df['denominator'], errors='coerce').fillna(0).astype(int)
        df['pct'] = pd.to_numeric(df['pct'], errors='coerce').fillna(0.0)
        dfs.append(df)
    except Exception as e:
        print(f"Error reading {f}: {e}")

if not dfs:
    print("No valid class_counts TSVs found; aborting plots.")
    raise SystemExit(0)

cc = pd.concat(dfs, ignore_index=True)

# Ensure directions are the exact expected labels
valid_dirs = ['Ensembl_to_CAT', 'CAT_to_Ensembl']
cc = cc[cc['direction'].isin(valid_dirs)].copy()

# Plot 1: Bar plots of median pct per class_code, one per direction
for d in valid_dirs:
    sub = cc[cc['direction']==d].copy()
    if sub.empty:
        print(f"No rows for {d}")
        continue
    med = (sub.groupby('class_code', as_index=False)['pct']
               .median()
               .sort_values('class_code'))
    plt.figure(figsize=(10,5))
    sns.barplot(data=med, x='class_code', y='pct', color='steelblue')
    plt.title(f"gffcompare class codes — {d} (Median % of query transcripts)")
    plt.ylabel('% of query transcripts')
    plt.xlabel('class_code')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    out = FIG_DIR / f"gffcompare_median_pct_{d}.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"Saved {out}")

# Plot 2: Boxplots of denominators per direction
plt.figure(figsize=(8,5))
order = valid_dirs
sns.boxplot(data=cc.drop_duplicates(['assembly_accession','sample_name','direction','denominator']),
            x='direction', y='denominator', order=order)
plt.title('gffcompare denominators (query transcripts) by direction')
plt.ylabel('Number of query transcripts (denominator)')
plt.xlabel('Direction')
plt.tight_layout()
out = FIG_DIR / 'gffcompare_denominator_boxplots.png'
plt.savefig(out, dpi=150)
plt.close()
print(f"Saved {out}")

# Plot 3 (optional): stacked bar for a sample of assemblies per direction
# Take up to 12 assemblies for legibility
sample_acc = (cc.groupby('assembly_accession')['denominator']
                .sum()
                .sort_values(ascending=False)
                .head(12)
                .index.tolist())
stack = cc[cc['assembly_accession'].isin(sample_acc)].copy()
for d in valid_dirs:
    sub = stack[stack['direction']==d].copy()
    if sub.empty:
        continue
    pivot = sub.pivot_table(index='assembly_accession', columns='class_code', values='pct', aggfunc='sum').fillna(0)
    pivot = pivot[pivot.columns.sort_values()]
    ax = pivot.plot(kind='bar', stacked=True, figsize=(12,6), colormap='tab20')
    ax.set_title(f'Stacked % by class_code — {d} (Percent of query transcripts)')
    ax.set_ylabel('% of query transcripts')
    ax.set_xlabel('assembly_accession')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    out = FIG_DIR / f"gffcompare_stacked_pct_{d}.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"Saved {out}")
